# CMU SCS News: collect AI-tagged links

Target: `https://www.cs.cmu.edu/news/archive?keywords=artificial-intelligence` (JavaScript-rendered, so Playwright + Chromium drives the archive's dynamic loader).

Key rules: loading progress is measured over **all** news URLs in the DOM; output is restricted to the exact CMU metadata tag `Artificial Intelligence`; zero new AI records is never by itself a stopping condition.

Output: `./Data/CMU SCS AI Articles Links.csv`


## One-time setup

Run the next cell once. If Playwright is already installed, pip will reuse the
existing installation.

In [ ]:
import subprocess
import sys

packages = [
    "playwright>=1.40",
    "pandas>=2.0",
    "tqdm>=4.66",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *packages],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "playwright", "install", "chromium"],
    check=True,
)

print("Playwright setup completed.")

In [ ]:
from pathlib import Path
import asyncio
import random
import re
from urllib.parse import (
    parse_qsl,
    urlencode,
    urljoin,
    urlsplit,
    urlunsplit,
)

import pandas as pd
from playwright.async_api import (
    Error as PlaywrightError,
    TimeoutError as PlaywrightTimeoutError,
    async_playwright,
)
from tqdm.auto import tqdm

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
START_URL = (
    "https://www.cs.cmu.edu/news/archive"
    "?keywords=artificial-intelligence"
)

TARGET_KEYWORD = "Artificial Intelligence"

DATA_DIR = Path("./Data")
OUTPUT_PATH = DATA_DIR / "CMU SCS AI Articles Links.csv"
DEBUG_DIR = DATA_DIR / "cmu_scs_ai_debug"

# Use False if you want to watch CMU load the archive.
HEADLESS = True

NAVIGATION_TIMEOUT_MS = 90_000
INITIAL_SETTLE_MS = 7_000
AFTER_CLICK_SETTLE_MS = 3_000

MAX_LOAD_MORE_CLICKS = 500

# A stalled round means clicking/loading produced no new general-news URL.
MAX_STALLED_LOADS = 8

# Before declaring that no loader exists, scroll and retry discovery.
CONTROL_DISCOVERY_RETRIES = 5
CONTROL_DISCOVERY_WAIT_MS = 2_000

DELAY_RANGE_SECONDS = (0.8, 1.5)
VIEWPORT = {"width": 1440, "height": 1000}

DATA_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def clean_text(value):
    if value is None:
        return ""
    return " ".join(str(value).split())


def hostname_of(url):
    return urlsplit(url).netloc.lower().split(":")[0]


def is_cmu_domain(url):
    host = hostname_of(url)
    return host == "cmu.edu" or host.endswith(".cmu.edu")


def canonicalize_article_url(href, base_url=START_URL):
    """Normalize article URLs and remove common tracking parameters."""
    if not href:
        return ""

    absolute = urljoin(base_url, href.strip())
    parts = urlsplit(absolute)

    if parts.scheme not in {"http", "https"} or not parts.netloc:
        return ""

    host = parts.netloc.lower()
    if host == "cs.cmu.edu":
        host = "www.cs.cmu.edu"

    path = re.sub(r"/{2,}", "/", parts.path)

    kept_query = []
    for key, value in parse_qsl(parts.query, keep_blank_values=True):
        lowered = key.casefold()

        if (
            lowered.startswith("utm_")
            or lowered in {
                "fbclid",
                "gclid",
                "mc_cid",
                "mc_eid",
            }
        ):
            continue

        kept_query.append((key, value))

    return urlunsplit(
        (
            parts.scheme.lower(),
            host,
            path,
            urlencode(kept_query, doseq=True),
            "",
        )
    )


def parse_keyword_list(value):
    return [
        clean_text(item)
        for item in clean_text(value).split(",")
        if clean_text(item)
    ]

In [ ]:
async def launch_browser(playwright):
    """Use installed Google Chrome first, then Playwright Chromium."""
    launch_options = {
        "headless": HEADLESS,
        "args": [
            "--disable-dev-shm-usage",
            "--no-first-run",
            "--no-default-browser-check",
        ],
    }

    try:
        browser = await playwright.chromium.launch(
            channel="chrome",
            **launch_options,
        )
        print("Using installed Google Chrome.")
        return browser
    except PlaywrightError as exc:
        print(
            "Installed Google Chrome was unavailable; "
            "using Playwright Chromium."
        )
        print(f"Chrome launch detail: {exc}")

    return await playwright.chromium.launch(**launch_options)


async def open_archive(page):
    """
    Open the archive and wait for cards to exist in the DOM.

    Important: the CMU page keeps both visible and hidden news cards in the
    document. Playwright's default wait_for_selector state is "visible", and
    with multiple matches it may repeatedly inspect the first hidden card.
    We therefore wait for an attached card count greater than zero instead.
    """
    response = await page.goto(
        START_URL,
        wait_until="domcontentloaded",
        timeout=NAVIGATION_TIMEOUT_MS,
    )

    await page.wait_for_function(
        """() => {
            return document.querySelectorAll(
                '.rsNewsArchiveItem'
            ).length > 0;
        }""",
        timeout=NAVIGATION_TIMEOUT_MS,
    )

    # Give CMU's filtering and dynamic-news scripts time to initialize.
    await page.wait_for_timeout(INITIAL_SETTLE_MS)

    status = response.status if response else None

    if status == 403:
        raise RuntimeError("CMU returned HTTP 403 in the browser.")

    card_count = await page.locator(
        ".rsNewsArchiveItem"
    ).count()

    print(
        f"Archive loaded with {card_count} "
        "news cards attached to the DOM."
    )

    return status

async def try_activate_ai_filter(page):
    """Best-effort activation; exact local filtering is still applied later."""
    labels = page.locator(
        "#rsNewsFilterControls label"
    ).filter(
        has_text=re.compile(
            r"^\s*Artificial Intelligence\s*$",
            re.IGNORECASE,
        )
    )

    if await labels.count() == 0:
        print(
            "No explicit AI filter label was found. "
            "Continuing with exact data-keywords filtering."
        )
        return False

    label = labels.first

    try:
        checkbox = label.locator("input[type='checkbox']")

        if await checkbox.count() > 0:
            if not await checkbox.first.is_checked():
                await label.click()
                await page.wait_for_timeout(AFTER_CLICK_SETTLE_MS)
                print("Activated the visible Artificial Intelligence filter.")
            else:
                print("Artificial Intelligence filter was already active.")
        else:
            await label.click()
            await page.wait_for_timeout(AFTER_CLICK_SETTLE_MS)
            print("Clicked the Artificial Intelligence filter label.")

        return True

    except Exception as exc:
        print(
            "Could not manipulate the visible filter control; "
            "local exact-tag filtering remains active."
        )
        print(f"Filter detail: {exc}")
        return False

In [ ]:
CARD_EXTRACTION_JS = r"""
() => {
  function clean(value) {
    return (value || "").replace(/\s+/g, " ").trim();
  }

  return Array.from(
    document.querySelectorAll(".rsNewsArchiveItem")
  ).map((card, index) => {
    const anchor = card.querySelector(
      ".articleText h2 a[href], h2 a[href], h3 a[href]"
    );
    const author = card.querySelector(".author");
    const date = card.querySelector(".date, time");
    const label = card.querySelector(
      ".articleLabels .rsTag, .rsTag"
    );
    const image = card.querySelector("img");

    return {
      dom_index: index,
      title: clean(anchor?.innerText || anchor?.textContent),
      href: anchor?.href || "",
      author: clean(author?.innerText || author?.textContent),
      displayed_date: clean(
        date?.getAttribute("datetime") ||
        date?.innerText ||
        date?.textContent
      ).replace(/^[-–—]\s*/, ""),
      label: clean(label?.innerText || label?.textContent),
      data_type: clean(card.getAttribute("data-type")),
      data_keywords: clean(card.getAttribute("data-keywords")),
      data_feeds: clean(card.getAttribute("data-feeds")),
      image_url: image?.src || "",
      image_alt: clean(image?.getAttribute("alt"))
    };
  });
}
"""


def normalize_cards(raw_cards, collection_round):
    """Keep exact CMU AI-tagged records and deduplicate by URL."""
    records = {}

    for raw in raw_cards:
        title = clean_text(raw.get("title", ""))
        url = canonicalize_article_url(raw.get("href", ""))

        if not title or not url:
            continue

        keywords = parse_keyword_list(
            raw.get("data_keywords", "")
        )

        # Exact site-metadata match; no title guessing.
        if TARGET_KEYWORD not in keywords:
            continue

        record = {
            "Article": title,
            "Link": url,
            "Listing Published Date": clean_text(
                raw.get("displayed_date", "")
            ),
            "Listing Author": clean_text(
                raw.get("author", "")
            ),
            "Listing Category": clean_text(
                raw.get("label", "")
            ),
            "Listing Type": clean_text(
                raw.get("data_type", "")
            ),
            "Listing Keywords": "; ".join(keywords),
            "Listing Feeds": "; ".join(
                parse_keyword_list(
                    raw.get("data_feeds", "")
                )
            ),
            "Image URL": canonicalize_article_url(
                raw.get("image_url", "")
            ),
            "Image Alt": clean_text(
                raw.get("image_alt", "")
            ),
            "AI Keyword Exact": True,
            "Link Domain": hostname_of(url),
            "Official CMU Domain": is_cmu_domain(url),
            "Collection Round": collection_round,
            "Archive URL": START_URL,
        }

        records[url] = record

    return records


def all_story_urls(raw_cards):
    """Return every unique story URL in the current DOM, AI or otherwise."""
    urls = set()

    for raw in raw_cards:
        url = canonicalize_article_url(
            raw.get("href", "")
        )

        if url:
            urls.add(url)

    return urls


def earliest_visible_date(records_by_url):
    """
    Return a diagnostic string only. Dates are intentionally not parsed here
    because the listing mixes several display formats.
    """
    dates = [
        clean_text(record.get("Listing Published Date", ""))
        for record in records_by_url.values()
        if clean_text(record.get("Listing Published Date", ""))
    ]

    return dates[-1] if dates else ""

In [ ]:
LOAD_MORE_DISCOVERY_JS = r"""
() => {
  function textOf(element) {
    return (
      element.innerText ||
      element.textContent ||
      element.value ||
      element.getAttribute("aria-label") ||
      element.getAttribute("title") ||
      ""
    ).replace(/\s+/g, " ").trim();
  }

  function visible(element) {
    const style = window.getComputedStyle(element);
    const rect = element.getBoundingClientRect();

    return (
      style.display !== "none" &&
      style.visibility !== "hidden" &&
      style.opacity !== "0" &&
      rect.width > 0 &&
      rect.height > 0
    );
  }

  const elements = Array.from(
    document.querySelectorAll(
      "button, a, input, [role='button']"
    )
  );

  const matches = elements
    .map((element) => {
      const text = textOf(element);
      const combined = [
        text,
        element.id || "",
        element.className || "",
        element.getAttribute("data-action") || "",
        element.getAttribute("data-load-action") || ""
      ].join(" ").toLowerCase();

      return {
        element,
        text,
        visible: visible(element),
        matches:
          combined.includes("load more news") ||
          combined.includes("load-more-news") ||
          combined.includes("loadmorenews") ||
          (
            combined.includes("load more") &&
            combined.includes("news")
          )
      };
    })
    .filter((item) => item.matches);

  return matches.map((item) => ({
    text: item.text,
    visible: item.visible,
    tag: item.element.tagName,
    id: item.element.id || "",
    className: String(item.element.className || "")
  }));
}
"""


async def loader_diagnostics(page):
    return await page.evaluate(
        LOAD_MORE_DISCOVERY_JS
    )


async def click_load_more_with_retries(page):
    """
    Scroll to the archive bottom and click a visible Load More control.

    Returns:
        clicked: bool
        diagnostics: list[dict]
    """
    last_diagnostics = []

    for attempt in range(
        1,
        CONTROL_DISCOVERY_RETRIES + 1,
    ):
        await page.evaluate(
            """() => {
                window.scrollTo(
                    0,
                    document.documentElement.scrollHeight
                );
            }"""
        )

        await page.wait_for_timeout(
            CONTROL_DISCOVERY_WAIT_MS
        )

        diagnostics = await loader_diagnostics(page)
        last_diagnostics = diagnostics

        result = await page.evaluate(
            r"""() => {
              function textOf(element) {
                return (
                  element.innerText ||
                  element.textContent ||
                  element.value ||
                  element.getAttribute("aria-label") ||
                  element.getAttribute("title") ||
                  ""
                ).replace(/\s+/g, " ").trim();
              }

              function visible(element) {
                const style = window.getComputedStyle(element);
                const rect = element.getBoundingClientRect();

                return (
                  style.display !== "none" &&
                  style.visibility !== "hidden" &&
                  style.opacity !== "0" &&
                  rect.width > 0 &&
                  rect.height > 0
                );
              }

              const elements = Array.from(
                document.querySelectorAll(
                  "button, a, input, [role='button']"
                )
              );

              const candidate = elements.find((element) => {
                const text = textOf(element);
                const combined = [
                  text,
                  element.id || "",
                  element.className || "",
                  element.getAttribute("data-action") || "",
                  element.getAttribute("data-load-action") || ""
                ].join(" ").toLowerCase();

                const matches =
                  combined.includes("load more news") ||
                  combined.includes("load-more-news") ||
                  combined.includes("loadmorenews") ||
                  (
                    combined.includes("load more") &&
                    combined.includes("news")
                  );

                return matches && visible(element);
              });

              if (!candidate) {
                return {
                  clicked: false,
                  text: "",
                  tag: "",
                  id: "",
                  className: ""
                };
              }

              candidate.scrollIntoView({
                block: "center"
              });
              candidate.click();

              return {
                clicked: true,
                text: textOf(candidate),
                tag: candidate.tagName,
                id: candidate.id || "",
                className: String(
                  candidate.className || ""
                )
              };
            }"""
        )

        if result.get("clicked"):
            print(
                "Clicked loader "
                f"(attempt {attempt}): "
                f"{result}"
            )
            return True, diagnostics

    return False, last_diagnostics


async def save_debug_files(page, suffix):
    html_path = DEBUG_DIR / f"{suffix}.html"
    png_path = DEBUG_DIR / f"{suffix}.png"
    text_path = DEBUG_DIR / f"{suffix}.txt"

    html_path.write_text(
        await page.content(),
        encoding="utf-8",
    )

    await page.screenshot(
        path=str(png_path),
        full_page=True,
    )

    diagnostics = await loader_diagnostics(page)

    card_count = await page.locator(
        ".rsNewsArchiveItem"
    ).count()

    text_path.write_text(
        "\n".join(
            [
                f"URL: {page.url}",
                f"Title: {await page.title()}",
                f"Cards: {card_count}",
                f"Scroll height: {await page.evaluate('document.documentElement.scrollHeight')}",
                "",
                "LOAD-MORE CANDIDATES:",
                *[
                    str(item)
                    for item in diagnostics
                ],
            ]
        ),
        encoding="utf-8",
    )

    print("Saved diagnostics:")
    print(f"  {html_path.resolve()}")
    print(f"  {png_path.resolve()}")
    print(f"  {text_path.resolve()}")

In [ ]:
async def collect_all_ai_links():
    ai_records_by_url = {}
    seen_all_story_urls = set()

    stalled_loads = 0
    stop_reason = ""

    async with async_playwright() as playwright:
        browser = await launch_browser(playwright)

        try:
            context = await browser.new_context(
                viewport=VIEWPORT,
                locale="en-US",
                timezone_id="America/New_York",
            )
            page = await context.new_page()

            status = await open_archive(page)
            print(f"Archive HTTP status: {status}")
            print(f"Loaded URL: {page.url}")
            print(
                "AI selection mode: exact local match on "
                f"data-keywords = {TARGET_KEYWORD!r}"
            )

            # The filtered URL normally initializes the correct UI state.
            await try_activate_ai_filter(page)

            for round_number in tqdm(
                range(MAX_LOAD_MORE_CLICKS + 1),
                desc="CMU load-more rounds",
            ):
                raw_cards = await page.evaluate(
                    CARD_EXTRACTION_JS
                )

                current_all_urls = all_story_urls(raw_cards)

                before_all = len(seen_all_story_urls)
                seen_all_story_urls.update(
                    current_all_urls
                )
                new_general_urls = (
                    len(seen_all_story_urls) - before_all
                )

                round_ai_records = normalize_cards(
                    raw_cards,
                    collection_round=round_number,
                )

                before_ai = len(ai_records_by_url)
                ai_records_by_url.update(
                    round_ai_records
                )
                new_ai_urls = (
                    len(ai_records_by_url) - before_ai
                )

                print(
                    f"Round {round_number}: "
                    f"{len(raw_cards)} DOM cards; "
                    f"{new_general_urls} new general URLs; "
                    f"{len(seen_all_story_urls)} general total; "
                    f"{new_ai_urls} new AI URLs; "
                    f"{len(ai_records_by_url)} AI total."
                )

                checkpoint = (
                    pd.DataFrame(
                        ai_records_by_url.values()
                    )
                    .drop_duplicates(subset="Link")
                    .reset_index(drop=True)
                )

                checkpoint.to_csv(
                    OUTPUT_PATH,
                    index=False,
                    encoding="utf-8-sig",
                )

                before_click_urls = set(
                    current_all_urls
                )

                clicked, loader_info = (
                    await click_load_more_with_retries(
                        page
                    )
                )

                if not clicked:
                    stop_reason = (
                        "No visible Load More News control "
                        "was found after scrolling and retries."
                    )

                    print(f"Stopped: {stop_reason}")
                    print(
                        "Loader candidates found:",
                        loader_info,
                    )

                    await save_debug_files(
                        page,
                        "no_loader_filtered_archive",
                    )
                    break

                await page.wait_for_timeout(
                    AFTER_CLICK_SETTLE_MS
                )

                # Wait until at least one new URL appears. The DOM count may
                # remain constant when CMU replaces old hidden cards.
                try:
                    await page.wait_for_function(
                        """previousUrls => {
                            const currentUrls = Array.from(
                                document.querySelectorAll(
                                  '.rsNewsArchiveItem h2 a[href], ' +
                                  '.rsNewsArchiveItem h3 a[href]'
                                )
                            ).map(
                                anchor => anchor.href
                            );

                            return currentUrls.some(
                                url => !previousUrls.includes(url)
                            );
                        }""",
                        arg=list(before_click_urls),
                        timeout=25_000,
                    )
                except PlaywrightTimeoutError:
                    pass

                await page.wait_for_timeout(
                    int(
                        random.uniform(
                            *DELAY_RANGE_SECONDS
                        )
                        * 1000
                    )
                )

                after_raw_cards = await page.evaluate(
                    CARD_EXTRACTION_JS
                )
                after_urls = all_story_urls(
                    after_raw_cards
                )

                genuinely_new_after_click = len(
                    after_urls - seen_all_story_urls
                )

                if genuinely_new_after_click == 0:
                    stalled_loads += 1
                else:
                    stalled_loads = 0

                print(
                    "Post-click progress: "
                    f"{genuinely_new_after_click} URLs not "
                    "previously seen; "
                    f"stalled={stalled_loads}/"
                    f"{MAX_STALLED_LOADS}."
                )

                if stalled_loads >= MAX_STALLED_LOADS:
                    stop_reason = (
                        "The loader was clicked repeatedly but "
                        "added no new general-news URLs."
                    )

                    await save_debug_files(
                        page,
                        "stalled_filtered_archive",
                    )

                    print(f"Stopped: {stop_reason}")
                    break

            else:
                stop_reason = (
                    "Reached MAX_LOAD_MORE_CLICKS."
                )

                await save_debug_files(
                    page,
                    "max_click_limit_filtered_archive",
                )

                print(f"Stopped: {stop_reason}")

        finally:
            await browser.close()

    result = (
        pd.DataFrame(
            ai_records_by_url.values()
        )
        .drop_duplicates(subset="Link")
        .reset_index(drop=True)
    )

    if not result.empty:
        result["Crawler Stop Reason"] = stop_reason
        result["General News URLs Observed"] = (
            len(seen_all_story_urls)
        )

    print(
        f"Final: observed {len(seen_all_story_urls)} "
        f"general-news URLs and retained "
        f"{len(result)} exact AI-tagged URLs."
    )

    return result

In [ ]:
article_links = await collect_all_ai_links()

article_links.head()

In [ ]:
article_links.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    f"Saved {len(article_links)} unique AI-tagged links to: "
    f"{OUTPUT_PATH.resolve()}"
)

In [ ]:
# Quality checks.
if not article_links.empty:
    print("Duplicate URLs:", article_links["Link"].duplicated().sum())

    print(
        "Official CMU links:",
        article_links["Official CMU Domain"].sum(),
    )

    print(
        "External links:",
        (~article_links["Official CMU Domain"]).sum(),
    )

    print(
        "Listing categories:",
        article_links["Listing Category"]
        .value_counts(dropna=False)
        .to_dict(),
    )

    invalid = ~article_links["Listing Keywords"].fillna("").apply(
        lambda value: TARGET_KEYWORD in [
            item.strip()
            for item in value.split(";")
        ]
    )

    print(
        "Rows failing exact AI-tag check:",
        invalid.sum(),
    )

    print(
        "General news URLs observed:",
        article_links["General News URLs Observed"].max(),
    )

    print(
        "Crawler stop reason:",
        article_links["Crawler Stop Reason"].iloc[0],
    )

    date_check = pd.to_datetime(
        article_links["Listing Published Date"]
        .str.replace(
            r"^[-–—]\s*",
            "",
            regex=True,
        ),
        errors="coerce",
    )

    print(
        "Newest parsed date:",
        date_check.max(),
    )
    print(
        "Oldest parsed date:",
        date_check.min(),
    )